# Assignment 3: Building a Dungeon Master Agent 🐉

## Overview
In this assignment, you will explore the architecture of **AI Agents**. Unlike simple chatbots, Agents have the ability to "reason" and use "tools" to interact with the world.

We will build a **ReAct (Reason + Act) Agent** from scratch.
**The Goal:** Create a "Dungeon Master" (DM) agent that narrates a text adventure. Crucially, when a player attempts a risky action (like attacking a goblin), the Agent must pause, use a **Dice Roll Tool** to determine the outcome, and then narrate the result based on that roll.

### Concepts Covered:
1.  **Tool Definition:** Creating Python functions the AI can "call".
2.  **Prompt Engineering:** Teaching the AI the protocol for thinking and acting.
3.  **The Agent Loop:** The logic that parses AI output, executes tools, and feeds the result back.

---
**Note:** This assignment uses a `MockLLM` class to simulate an AI response so you don"t need an API key to run the logic tests. However, the logic you build is exactly what you would use with GPT-4, Claude, or Llama.

In [ ]:
# Setup - Run this cell first!
import re
import random
import time

print("Setup complete. Ready to build an agent!")

## Part 1: The Tools and The Brain 🧠

An agent needs two things before it can start:
1.  **Tools:** Capabilities it has (functions it can call).
2.  **Instructions:** A System Prompt telling it how to behave and how to use those tools.

### Task 1.1: Define the Dice Tool
Our DM needs to roll dice to check for success/failure.

**TODO:** Implement the `roll_dice` function below. It should take an input string (like "d20" or just ignore input for simplicity and assume d20) and return a random integer between 1 and 20.

In [ ]:
def roll_dice(sides=20):
    """
    Simulates rolling a die.
    Args:
        sides (int): The number of sides on the die (default 20).
    Returns:
        int: A random number between 1 and "sides".
    """
    # TODO: Implement this function
    pass

In [ ]:
# TEST YOUR CODE
print(f"Rolling a d20: {roll_dice(20)}")

# Verification
results = [roll_dice(20) for _ in range(100)]
assert all(1 <= r <= 20 for r in results if r is not None), "Error: Dice roll out of bounds or None!"
assert len(set(results)) > 1, "Error: The dice seems to always return the same number!"
print("✅ Task 1.1 Passed!")

### Task 1.2: The System Prompt

We need to tell the LLM how to "think". We use the **ReAct** pattern:
1.  **Thought:** The agent reasons about what to do.
2.  **Action:** The agent picks a tool to use.
3.  **Observation:** The result of the tool (fed back by code).

**TODO:** Fill in the `SYSTEM_PROMPT` string. It must explicitly tell the Agent to use the format:
```
Thought: <your reasoning>
Action: roll_dice
```
if it needs to check for success. If no tool is needed, it should just reply.

In [ ]:
# TODO: Write a system prompt that instructs the agent to act as a Dungeon Master.
# Key Requirement: Tell it that if a result is uncertain, it MUST output "Action: roll_dice".

SYSTEM_PROMPT = """
You are a text adventure Dungeon Master.
Your goal is to narrate an exciting story.
...
"""

In [ ]:
# TEST YOUR CODE
assert "Action:" in SYSTEM_PROMPT, "Prompt must mention the "Action:" keyword"
assert "roll_dice" in SYSTEM_PROMPT, "Prompt must mention the "roll_dice" tool"
print("✅ Task 1.2 Passed! (Prompt content looked good)")

## Part 2: The Agent Loop 🔄

This is the heart of the agent. The loop works like this:
1.  Take User Input.
2.  Ask LLM for response.
3.  **Parse** response: Did it ask for an Action?
4.  **If Action:** Execute Tool -> Get Result -> Send Result back to LLM -> Repeat.
5.  **If No Action:** Show response to user.

### Task 2.1: The Parser
We need a helper function to detect if the LLM wants to use a tool.

In [ ]:
def parse_response(llm_output):
    """
    Parses the LLM"s output string to look for "Action: roll_dice".

    Returns:
        bool: True if tool is requested, False otherwise.
    """
    # TODO: Check if the string contains "Action: roll_dice" (case-insensitive is better)
    pass

In [ ]:
# TEST YOUR CODE
test_str_1 = "Thought: The user is attacking. Action: roll_dice"
test_str_2 = "You walk down the hallway. It is quiet."

assert parse_response(test_str_1) == True, "Failed to detect action."
assert parse_response(test_str_2) == False, "False positive detected."
print("✅ Task 2.1 Passed!")

### Task 2.2: The Execution Loop

Now we assemble the `run_turn` function.

To make this work without an API key, we will use `MockLLM`.
- If you send it text containing "Action: roll_dice", it simulates the "Observation" step.
- If you send it normal text, it simulates the "Reasoning" step.

**TODO:** Complete the `run_turn` function.

In [ ]:
class MockLLM:
    """A fake LLM that responds predictably for testing logic."""
    def generate(self, prompt):
        # If the prompt shows we just rolled a die (Observation), the LLM narrates the result
        if "Observation:" in prompt:
            return "The die rolls... a critical hit! The goblin falls."

        # If the user says "attack", the LLM decides to roll
        if "attack" in prompt.lower():
            return "Thought: The player is attacking. I need to check if they hit. Action: roll_dice"

        # Default conversation
        return "What do you want to do next?"

llm = MockLLM()

def run_turn(user_input, history=[]):
    """
    Executes a single turn of the agent.
    """
    # Step 1: Add user input to context
    context = "\n".join(history) + f"\nUser: {user_input}\n"

    # Step 2: First call to LLM
    response = llm.generate(context)
    print(f"🤖 Agent: {response}")

    # Step 3: Check for tools (TODO)
    # TODO: Use your parse_response function.
    # If it returns True:
    #    1. Call roll_dice()
    #    2. Create an observation string: f"Observation: {result}"
    #    3. Append this to context
    #    4. Call llm.generate(context) again to get the final narrative
    #    5. Update "response" with this new narrative

    # (Write your code here)

    return response

In [ ]:
# TEST YOUR CODE
print("--- Test 1: Simple Chat ---")
output_simple = run_turn("Hello?")
assert "What do you want" in output_simple, "Simple chat failed."

print("\n--- Test 2: Action Trigger ---")
# This input "attack" should trigger the MockLLM to say "Action: roll_dice"
# Your loop should catch that, roll (internally), and get the final "goblin falls" message.
output_action = run_turn("I attack the goblin!")

assert "goblin falls" in output_action, "The agent loop didn"t handle the tool execution correctly!"
print("\n✅ Task 2.2 Passed! You have built a working Agent Loop.")

## Conclusion
You have successfully built the core logic of an Autonomous Agent!

1.  It has a **System Prompt** to guide behavior.
2.  It has **Tools** (dice) to interact with the world.
3.  It has a **Control Loop** to perceive -> think -> act.

If you connected this to a real API (like OpenAI), the `MockLLM` would be replaced by a real brain, and you could play an infinite D&D campaign where the DM actually rolls dice for you!